# Final-sprint ladder-alignment research

**September 29–30, 2026**

This executed notebook summarizes the research arc after public checkpoint `8fa7538`. The central result is a validator correction: the older fixed-opponent panel had saturated and no longer tracked live ladder behavior. The replacement evaluator reconstructs recent keeper ladder episodes exactly before interpreting counterfactual policies.

All values below come from verified return bundles. Negative experiments are preserved rather than hidden.

In [1]:
import json
from pathlib import Path
import pandas as pd
import plotly.express as px

report = json.loads((Path('..') / 'reports' / 'final_sprint_2026-09-30.json').read_text())
rows = []
for name, item in report['milestones'].items():
    rows.append({'milestone': name, 'status': item.get('status'), 'decision': item.get('decision') or item.get('scientific_decision'), 'official_engine_games': item.get('new_official_engine_games', item.get('branch_and_calibration_games', 0))})
df = pd.DataFrame(rows)
df


                          milestone     status                       decision  official_engine_games
0               late_layer_ablation  COMPLETED                REJECTED_SCREEN                    234
1               onpolicy_population  COMPLETED              TRAINING_REJECTED                    448
2                   frontier_routes  COMPLETED                SCREEN_REJECTED                    428
3          public_adaptive_frontier  COMPLETED                SCREEN_REJECTED                    300
4                  ladder_alignment  COMPLETED  NO_FIXED_POLICY_IMPROVEMENT                      0
5                 boundary_selector  COMPLETED      INNER_VALIDATION_REJECTED                   1084
6                  boundary_rule_cv  COMPLETED            ROLLING_CV_REJECTED                      0
7  native_harvest_transport_failure     FAILED                           None                      0

In [2]:
fig = px.bar(df, x='milestone', y='official_engine_games', hover_data=['status', 'decision'], title='Official-engine work by final-sprint milestone')
fig.update_layout(width=1000, height=500, xaxis_tickangle=-35)
fig.show()


## Ladder-alignment correction

The replay-aligned validator requires exact terminal reward parity on the keeper before a counterfactual policy is scored. On the 48 recent keeper episodes used in the sprint, the keeper had **18 wins, 10 ties, and 20 losses (47.9% win/tie score)**. The older 24-episode development slice was **9 wins, 4 ties, and 11 losses (45.8%)**.

That result explained the mismatch between near-perfect performance on the old local panel and much weaker live rating.

In [3]:
development = report['milestones']['ladder_alignment']['development']
plot = pd.DataFrame([{'policy': name, 'score': values['score_rate'], 'rescues': values.get('rescued_keeper_losses', 0), 'sacrifices': values.get('sacrificed_keeper_wins', 0)} for name, values in development.items() if name != 'keeper'])
fig2 = px.scatter(plot, x='sacrifices', y='rescues', size='score', text='policy', title='Real-ladder development: rescues versus sacrificed keeper wins')
fig2.update_traces(textposition='top center')
fig2.update_layout(width=900, height=520)
fig2.show()


## Interpretation

Fixed replacement policies were globally worse than the keeper, but some rescued isolated losses. This motivated boundary-level borrowing. The resulting 1,080 branch games contained hindsight rescue signal, yet the first learned selector and a later rolling-origin rule search both failed forward validation.

The project therefore closed fixed-expert borrowing and moved to project-authored native mechanisms on the keeper. The current registered direction is harvest/service recovery; its first attempt failed operationally after game completion because an older diagnostics transport queried an exited process. The repair changes transport ordering only, so no scientific conclusion is claimed yet.